# Homework 01C — Multiclass SVM from scratch

**Two sessions · vectorized hinge loss · manual gradient · four autograded TODOs**

Softmax and SVM use the same linear scores $XW+b$ but express different preferences. Here, the correct class need not absorb all probability; it must beat every competitor by a margin. You will expose that logic in NumPy and derive the gradient that enforces it.

> Based on Stanford CS231n's [linear-classification notes](https://cs231n.github.io/linear-classify/) and the multiclass formulation of [Crammer and Singer (2001)](https://jmlr.org/papers/v2/crammer01a.html).

In [ ]:
import pickle
import tarfile
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

SEED = 42
CLASS_NAMES = np.array(['airplane', 'automobile', 'bird', 'cat', 'deer',
                        'dog', 'frog', 'horse', 'ship', 'truck'])
print('NumPy', np.__version__)

## 1. Load CIFAR-10 and protect the test set

This notebook is self-contained. It downloads the official archive once, creates a fixed 49,000/1,000 train/validation split, and centers every split using the training mean only.

In [ ]:
def load_cifar10(cache_dir='data'):
    cache = Path(cache_dir)
    archive = cache / 'cifar-10-python.tar.gz'
    root = cache / 'cifar-10-batches-py'
    cache.mkdir(parents=True, exist_ok=True)
    if not root.exists():
        if not archive.exists():
            print('Downloading CIFAR-10 (about 163 MB)...')
            urllib.request.urlretrieve(
                'https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz', archive
            )
        with tarfile.open(archive, 'r:gz') as tar:
            tar.extractall(cache, filter='data')

    def read_batch(path):
        with open(path, 'rb') as handle:
            batch = pickle.load(handle, encoding='bytes')
        images = batch[b'data'].reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)
        return images, np.asarray(batch[b'labels'])

    parts = [read_batch(root / f'data_batch_{i}') for i in range(1, 6)]
    X_train = np.concatenate([p[0] for p in parts])
    y_train = np.concatenate([p[1] for p in parts])
    X_test, y_test = read_batch(root / 'test_batch')
    return X_train, y_train, X_test, y_test

X_all_images, y_all, X_test_images, y_test = load_cifar10()
order = np.random.default_rng(SEED).permutation(len(X_all_images))
train_idx, val_idx = order[:49000], order[49000:]
X_train_images, y_train = X_all_images[train_idx], y_all[train_idx]
X_val_images, y_val = X_all_images[val_idx], y_all[val_idx]

def prepare(images):
    return images.reshape(len(images), -1).astype(np.float32) / 255.0

X_train, X_val, X_test = map(prepare, [X_train_images, X_val_images, X_test_images])
pixel_mean = X_train.mean(axis=0, keepdims=True)
X_train -= pixel_mean
X_val -= pixel_mean
X_test -= pixel_mean
print(X_train.shape, X_val.shape, X_test.shape)

## TODO 1 — Initialize the model

Create $W\in\mathbb{R}^{D\times C}$ from a zero-mean Gaussian with standard deviation `weight_scale`, using `np.random.default_rng(seed)`. Initialize $b\in\mathbb{R}^{C}$ to zeros. Return `(W, b)`. Small random weights break symmetry while keeping initial scores controlled.

In [ ]:
def initialize_parameters(n_features, n_classes, weight_scale=1e-3, seed=42):
    # TODO 1: reproducible Gaussian W and zero b.
    raise NotImplementedError('TODO 1')

In [ ]:
def check_todo_1(fn):
    W, b = fn(2000, 4, weight_scale=.02, seed=7)
    W2, b2 = fn(2000, 4, weight_scale=.02, seed=7)
    assert W.shape == (2000, 4) and b.shape == (4,)
    np.testing.assert_array_equal(W, W2)
    np.testing.assert_array_equal(b, np.zeros(4))
    assert abs(W.mean()) < .002 and .017 < W.std() < .023
    print('✓ TODO 1 passed: shape, scale, zeros, and reproducibility')

check_todo_1(initialize_parameters)

## Scores and prediction are supplied

These are unchanged from Softmax. The loss—not the score function—defines what the learned weights should prefer.

In [ ]:
def scores(X, W, b):
    return X @ W + b

def predict(X, W, b):
    return np.argmax(scores(X, W, b), axis=1)

## 2. See one hinge loss before vectorizing

Suppose the correct class is `cat` with score 3.2, and two wrong scores are 5.1 and 2.8. With $\Delta=1$, their margins are $5.1-3.2+1=2.9$ and $2.8-3.2+1=0.6$: both violate the required gap. A wrong class scoring 1.0 has margin $-1.2$, clipped to zero, and contributes no loss or gradient.

In [ ]:
example_scores = np.array([5.1, 3.2, 2.8, 1.0])
correct_class = 1
raw_margins = example_scores - example_scores[correct_class] + 1.0
margins = np.maximum(0, raw_margins)
margins[correct_class] = 0
print('raw margins:', raw_margins)
print('hinge terms:', margins, 'loss:', margins.sum())

## TODO 2 — Vectorized multiclass hinge loss

For every row, gather the correct-class score, broadcast it against all class scores, add `delta`, clip below zero, and explicitly zero the correct-class column. Average over instances and add $\lambda\lVert W\rVert_2^2$. Do not loop over instances or classes. Return a scalar.

In [ ]:
def svm_loss(X, y, W, b, reg=0.0, delta=1.0):
    # TODO 2: fully vectorized multiclass hinge loss.
    raise NotImplementedError('TODO 2')

In [ ]:
def check_todo_2(fn):
    X = np.array([[1., 0.], [0., 1.]])
    y = np.array([0, 2])
    W = np.array([[2., 1., -1.], [0., 1., 2.]])
    b = np.zeros(3)
    loss = fn(X, y, W, b, reg=.1)
    # Data loss: first row 0, second row margins [0, 0, 0]; only L2 remains.
    np.testing.assert_allclose(loss, .1 * np.sum(W * W))
    bad_y = np.array([2, 0])
    np.testing.assert_allclose(fn(X, bad_y, W, b, reg=0), 6.0)
    print('✓ TODO 2 passed: margins, correct-class zeroing, averaging, and L2')

check_todo_2(svm_loss)

## TODO 3 — Backward from active violations

Create a coefficient matrix $G$ with `1` wherever an incorrect margin is positive. If row $i$ has $m_i$ active incorrect margins, put $-m_i$ in its correct-class position. Then

$$dW=\frac{1}{B}X^TG+2\lambda W,\qquad db=\frac{1}{B}\sum_iG_i.$$

Implement `svm_loss_and_gradients` and return `(loss, dW, db)`. You may call `svm_loss` for the scalar, but compute the vectorized margins and mask for the gradient.

In [ ]:
def svm_loss_and_gradients(X, y, W, b, reg=0.0, delta=1.0):
    # TODO 3: return loss, dW, db using the active-margin mask.
    raise NotImplementedError('TODO 3')

In [ ]:
def check_todo_3(fn):
    X = np.array([[1., 2.], [-1., 1.], [.5, -2.]])
    y = np.array([0, 2, 1])
    W = np.array([[.1, -.2, .3], [-.1, .2, .05]])
    b = np.array([.01, -.02, .03])
    loss, dW, db = fn(X, y, W, b, reg=.1)
    assert np.ndim(loss) == 0 and dW.shape == W.shape and db.shape == b.shape
    np.testing.assert_allclose(loss, svm_loss(X, y, W, b, reg=.1))
    np.testing.assert_allclose(db.sum(), 0, atol=1e-12)
    print('✓ TODO 3 structural checks passed; run the numerical checker next')

check_todo_3(svm_loss_and_gradients)

### Numerical gradient checker

Hinge loss is not differentiable exactly at margin zero. Random tiny inputs almost never land exactly there, so central differences should agree with the analytical gradient. If a check sits at a kink, changing the random seed distinguishes that expected ambiguity from a systematic bug.

In [ ]:
def numerical_gradient_check(fn, X, y, W, b, reg=.05, checks=12, h=1e-5, seed=1):
    _, dW, _ = fn(X, y, W, b, reg)
    rng = np.random.default_rng(seed)
    errors = []
    for _ in range(checks):
        index = tuple(rng.integers(size) for size in W.shape)
        old = W[index]
        W[index] = old + h
        plus = fn(X, y, W, b, reg)[0]
        W[index] = old - h
        minus = fn(X, y, W, b, reg)[0]
        W[index] = old
        numeric = (plus - minus) / (2 * h)
        analytic = dW[index]
        errors.append(abs(numeric - analytic) / max(1e-8, abs(numeric) + abs(analytic)))
    print(f'max relative error: {max(errors):.2e}')
    assert max(errors) < 1e-6
    print('✓ numerical gradient check passed')

tiny_rng = np.random.default_rng(4)
tiny_X = tiny_rng.normal(size=(6, 7))
tiny_y = np.array([0, 2, 1, 2, 0, 1])
tiny_W = tiny_rng.normal(scale=.1, size=(7, 3))
tiny_b = tiny_rng.normal(scale=.1, size=3)
numerical_gradient_check(svm_loss_and_gradients, tiny_X, tiny_y, tiny_W, tiny_b)

**Reflection 1.** Why does a class with zero margin contribute nothing to the gradient? Compare this with Softmax, where every class probability generally contributes.

## TODO 4 — Apply the backward result

Complete one in-place SGD step: $W\leftarrow W-\eta dW$ and $b\leftarrow b-\eta db$. This update is mechanically identical to Softmax; only the loss-generated gradient differs.

In [ ]:
def sgd_step(W, b, dW, db, learning_rate):
    # TODO 4: update both arrays in place and return them.
    raise NotImplementedError('TODO 4')

In [ ]:
def check_todo_4(fn):
    W = np.array([[1., -2.], [3., 4.]])
    b = np.array([.5, -.5])
    W_id, b_id = id(W), id(b)
    out_W, out_b = fn(W, b, np.ones_like(W), np.array([2., -1.]), .1)
    assert id(out_W) == W_id and id(out_b) == b_id
    np.testing.assert_allclose(W, [[.9, -2.1], [2.9, 3.9]])
    np.testing.assert_allclose(b, [.3, -.4])
    print('✓ TODO 4 passed: backward information updates both parameters')

check_todo_4(sgd_step)

## 3. Supplied mini-batch training loop

As in the Softmax notebook, the loop samples batches, runs forward and backward, applies an update, and records the loss. Read the composition before executing it.

In [ ]:
def train_svm(X, y, *, learning_rate=1e-3, reg=1e-4, epochs=8,
              batch_size=256, seed=42, verbose=False):
    W, b = initialize_parameters(X.shape[1], len(CLASS_NAMES), seed=seed)
    rng = np.random.default_rng(seed)
    history = []
    for epoch in range(epochs):
        order = rng.permutation(len(X))
        losses = []
        for start in range(0, len(X), batch_size):
            idx = order[start:start + batch_size]
            loss, dW, db = svm_loss_and_gradients(X[idx], y[idx], W, b, reg)
            sgd_step(W, b, dW, db, learning_rate)
            losses.append(loss)
        history.append(float(np.mean(losses)))
        if verbose:
            print(f'epoch {epoch + 1:02d} | loss {history[-1]:.4f}')
    return W, b, history

probe_W, probe_b, probe_loss = train_svm(
    X_train, y_train, learning_rate=1e-3, reg=1e-4, epochs=2, verbose=True
)
plt.plot([1, 2], probe_loss, marker='o')
plt.xlabel('epoch'); plt.ylabel('mean mini-batch loss')
plt.title('Two-epoch SVM diagnostic'); plt.xticks([1, 2]); plt.grid(alpha=.2)
plt.show()

**Reflection 2.** Does the loss fall? At initialization, why can one image contribute as many as nine positive hinge terms?

## 4. Supplied learning-rate × L2 search

The validation set chooses the optimization scale and regularization strength. Each candidate starts from the same random seed for a fairer comparison.

In [ ]:
def grid_search(X_train, y_train, X_val, y_val, learning_rates, regs, epochs=8):
    results, best = [], None
    for lr in learning_rates:
        for reg in regs:
            W, b, losses = train_svm(
                X_train, y_train, learning_rate=lr, reg=reg, epochs=epochs, seed=42
            )
            train_acc = float(np.mean(predict(X_train, W, b) == y_train))
            val_acc = float(np.mean(predict(X_val, W, b) == y_val))
            item = {'learning_rate': lr, 'reg': reg, 'W': W, 'b': b,
                    'losses': losses, 'train_accuracy': train_acc, 'val_accuracy': val_acc}
            results.append(item)
            if best is None or val_acc > best['val_accuracy']:
                best = item
            print(f'lr={lr:.0e} reg={reg:.0e} | train={train_acc:.3f} val={val_acc:.3f}')
    return results, best

LEARNING_RATES = [3e-4, 1e-3, 3e-3]
REG_STRENGTHS = [0.0, 1e-4, 1e-3]
results, best = grid_search(X_train, y_train, X_val, y_val, LEARNING_RATES, REG_STRENGTHS)
print('best:', {k: best[k] for k in ['learning_rate', 'reg', 'train_accuracy', 'val_accuracy']})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for item in results:
    label = f"lr={item['learning_rate']:.0e}, λ={item['reg']:.0e}"
    axes[0].plot(range(1, len(item['losses']) + 1), item['losses'], label=label)
axes[0].set(xlabel='epoch', ylabel='training loss', title='SVM optimization paths')
axes[0].grid(alpha=.2); axes[0].legend(fontsize=7, ncol=2)
for reg in REG_STRENGTHS:
    subset = [r for r in results if r['reg'] == reg]
    axes[1].plot(LEARNING_RATES, [r['val_accuracy'] for r in subset], marker='o', label=f'λ={reg:.0e}')
axes[1].set_xscale('log')
axes[1].set(xlabel='learning rate', ylabel='validation accuracy', title='Validation selects the candidate')
axes[1].grid(alpha=.2); axes[1].legend(); plt.tight_layout()

**Reflection 3.** Use the loss curves and validation scores to justify the chosen candidate. Compare the useful learning-rate scale with Softmax; why need the numerical values not match?

## 5. Final test and class templates

Use the test set once, then reshape each learned weight column into an image. Compare these templates directly with the Softmax notebook.

In [ ]:
test_accuracy = np.mean(predict(X_test, best['W'], best['b']) == y_test)
print(f"train accuracy:      {best['train_accuracy']:.3%}")
print(f"validation accuracy: {best['val_accuracy']:.3%}")
print(f'test accuracy:       {test_accuracy:.3%}')

weights = best['W'].reshape(32, 32, 3, 10)
display_weights = (weights - weights.min()) / (weights.max() - weights.min() + 1e-12)
fig, axes = plt.subplots(2, 5, figsize=(11, 5))
for label, ax in enumerate(axes.flat):
    ax.imshow(display_weights[:, :, :, label])
    ax.set_title(CLASS_NAMES[label]); ax.axis('off')
plt.suptitle('Multiclass SVM weight templates'); plt.tight_layout()

## Final comparison

1. Compare SVM and Softmax validation/test accuracy without declaring a universal winner.
2. Compare their weight templates: which visual structures are stable across both objectives?
3. Explain the different behavior of an incorrect class far below the correct score under hinge loss and Softmax loss.
4. Explain why the gradient row coefficients sum to zero for both losses.
5. Which code was reusable across the two notebooks, and which code changed because the learning objective changed?

## Submission checklist

- [ ] Restart kernel and run every cell.
- [ ] Four TODO checks and the numerical gradient check are green.
- [ ] Diagnostic, grid-search plots, and class templates are visible.
- [ ] Test data appears only after model selection.
- [ ] Reflections and final comparison are complete.